# Domain-Specific Q&A Chatbot – Travel Domain

**Name:** Muskan  
**Date:** March 2025 (or write today's date: e.g. 10 March 2025)  
**Task:** AI & Chatbot Developer Internship – Task 9

## Overview
This notebook implements a simple **retrieval-based Q&A chatbot** focused on the **travel domain**.

Features:
- Uses a small FAQ dataset (CSV format)
- Preprocesses text with NLTK
- Calculates similarity using **TF-IDF + Cosine Similarity** (scikit-learn)
- Provides answers via command-line interface
- Optional: Streamlit web interface

Technologies used:
- Python
- pandas, numpy
- scikit-learn
- NLTK
- spaCy (optional preprocessing)
- Streamlit (optional UI)

--- # 🤖 Travel Q&A Chatbot  

## Project Goal
Build a domain-specific chatbot that answers travel-related questions using:
- A custom FAQ knowledge base
- TF-IDF vectorization
- Cosine similarity matching

## Table of Contents
1. Setup & Imports  
2. Knowledge Base (FAQ dataset)  
3. Text Preprocessing  
4. Vectorization & Similarity Search  
5. Chatbot Logic  
6. Command-line Interface  
7. (Optional) Streamlit Web App  
8. Testing & Examples

---

In [27]:
import pandas as pd

# Sample FAQ for Travel domain (10-20 entries for a small knowledge base)
data = {
    'Question': [
        'What are the best places to visit in Europe?',
        'How do I book a flight?',
        'What is the visa requirement for USA?',
        'Best budget hotels in Paris?',
        'How to travel sustainably?',
        'What to pack for a beach vacation?',
        'How to handle jet lag?',
        'Top adventure activities in New Zealand?',
        'Currency exchange tips?',
        'Safety tips for solo travelers?'
    ],
    'Answer': [
        'Popular spots include Paris, Rome, London, and Barcelona. Plan based on season.',
        'Use websites like Kayak or Expedia, compare prices, and book directly with airlines.',
        'Most visitors need ESTA or a visa; check the official US embassy site.',
        'Options like Ibis or Hostelworld offer affordable stays under 100 EUR/night.',
        'Choose eco-friendly transport, reduce plastic, and support local businesses.',
        'Essentials: sunscreen, swimsuit, hat, light clothes, and flip-flops.',
        'Stay hydrated, adjust sleep schedule, and expose to natural light.',
        'Bungee jumping, hiking in Fiordland, and skydiving in Queenstown.',
        'Use apps like XE, avoid airport exchanges, and carry a mix of cash/cards.',
        'Share itinerary, stay in well-reviewed areas, and use ride-sharing apps.'
    ]
}

df = pd.DataFrame(data)
df.to_csv('travel_faq.csv', index=False)
print("Dataset created: travel_faq.csv")

Dataset created: travel_faq.csv


In [28]:
df = pd.read_csv('travel_faq.csv'); df.head()

,Question,Answer
0,What are the best places to visit in Europe?,"Popular spots include Paris, Rome, London, and..."
1,How do I book a flight?,"Use websites like Kayak or Expedia, compare pr..."
2,What is the visa requirement for USA?,Most visitors need ESTA or a visa; check the o...
3,Best budget hotels in Paris?,Options like Ibis or Hostelworld offer afforda...
4,How to travel sustainably?,"Choose eco-friendly transport, reduce plastic,..."


In [30]:
# ────────────────────────────────────────────────
# Cell: Text Preprocessing Functions
# ────────────────────────────────────────────────

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

# Download required NLTK data only if not already present
# (quiet=True prevents annoying download messages)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Prepare stop words once (global)
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Clean and preprocess text for TF-IDF / similarity:
    - lowercase
    - remove punctuation
    - tokenize
    - remove stopwords
    - join back to string
    """
    if not isinstance(text, str):
        return ""   # protect against NaN / None
    
    # Lowercase
    text = text.lower()
    
    # Remove punctuation (fast method)
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords & short tokens (optional but improves quality)
    tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    
    # Join back to string (what TF-IDF expects)
    return ' '.join(tokens)


# ─── Quick test ────────────────────────────────────────
# Run this part to verify everything works

test_sentences = [
    "What are the best places to visit in Europe?",
    "How do I book a flight quickly???",
    "BEST budget hotels in Paris!!!",
    None,                               # edge case
    "   "                               # empty
]

print("Preprocessing test:\n")
for sent in test_sentences:
    cleaned = preprocess_text(sent)
    print(f"Original: {sent}")
    print(f"Cleaned : {cleaned}\n")

[nltk_data] Error loading punkt: <urlopen error unknown url type:
[nltk_data]     https>
[nltk_data] Error loading stopwords: <urlopen error unknown url type:
[nltk_data]     https>


<class 'LookupError'>: 
**********************************************************************
  Resource [93mstopwords[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('stopwords')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/stopwords[0m

  Searched in:
    - '/home/pyodide/nltk_data'
    - '/nltk_data'
    - '/share/nltk_data'
    - '/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [31]:
# Next cell – Load FAQ and vectorize
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('travel_faq.csv')

# Apply preprocessing to all questions
df['Processed_Question'] = df['Question'].apply(preprocess_text)

# Create TF-IDF
vectorizer = TfidfVectorizer()
question_vectors = vectorizer.fit_transform(df['Processed_Question'])

print("Vectorization done. Shape:", question_vectors.shape)

<class 'LookupError'>: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/home/pyodide/nltk_data'
    - '/nltk_data'
    - '/share/nltk_data'
    - '/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load FAQ
df = pd.read_csv('travel_faq.csv')
df['Processed_Question'] = df['Question'].apply(preprocess_text)

# Vectorizer
vectorizer = TfidfVectorizer()
question_vectors = vectorizer.fit_transform(df['Processed_Question'])

<class 'LookupError'>: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/home/pyodide/nltk_data'
    - '/nltk_data'
    - '/share/nltk_data'
    - '/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [36]:
def get_best_answer(user_query):
    processed_query = preprocess_text(user_query)
    query_vector = vectorizer.transform([processed_query])
    similarities = cosine_similarity(query_vector, question_vectors)
    best_index = similarities.argmax()
    best_similarity = similarities[0][best_index]
    
    if best_similarity > 0.3:  # Threshold to avoid irrelevant matches; tune if needed
        return df['Answer'].iloc[best_index]
    else:
        return "Sorry, I don't have an answer for that. Try rephrasing!"

In [39]:
def chatbot_cli():
    print("Welcome to Travel Chatbot! Type 'exit' to quit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            print("Goodbye!")
            break
        response = get_best_answer(user_input)
        print(f"Bot: {response}")

# Run it
# chatbot_cli()  # Uncomment to test in notebook (use Kernel > Interrupt to stop)